# 05 — Pure Premium: Resolving the Mismatch and Combining Frequency × Severity

**Phase 7**, step 1 of 5: resolve the frequency/severity mismatch documented since Phase 3, and build the combined pure-premium table.

Recall from Phase 3: naively multiplying the frequency champion (Phase 5, fit on `ClaimNb` - *reported* claims) by the severity champion (Phase 6, fit on paid claim amounts) overstates true pure premium by **36.3%**, proven to equal `sum(ClaimNb)/sum(ClaimNbFromSev)`. This has been deferred through two phases; it gets resolved here, properly - not approximated.

In [1]:
import numpy as np
import pandas as pd

from auto_pricing.frequency import predict_frequency
from auto_pricing.severity import build_severity_table, fit_lognormal_model, predict_lognormal_severity
from auto_pricing.pure_premium import (
    fit_paid_frequency_glm,
    predict_annual_pure_premium,
    predict_expected_loss,
)

pp_table = pd.read_parquet("../data/processed/pure_premium_table.parquet")
train = pp_table[pp_table["split"] == "train"].copy()
print(f"train rows: {len(train):,}")
print(f"columns added beyond model_table: ClaimNbFromSev, ClaimAmountSum")

train rows: 474,609
columns added beyond model_table: ClaimNbFromSev, ClaimAmountSum


## 1. Why a flat correction factor was considered and rejected

In [2]:
overall_payment_rate = train["ClaimNbFromSev"].sum() / train["ClaimNb"].sum()
print(f"overall payment rate (train): {overall_payment_rate:.4f}")
print()

by_bonusmalus = train.groupby("BonusMalusBand", observed=True).apply(
    lambda d: d["ClaimNbFromSev"].sum() / d["ClaimNb"].sum() if d["ClaimNb"].sum() > 0 else float("nan"),
    include_groups=False,
)
by_drivage = train.groupby("DrivAgeBand", observed=True).apply(
    lambda d: d["ClaimNbFromSev"].sum() / d["ClaimNb"].sum() if d["ClaimNb"].sum() > 0 else float("nan"),
    include_groups=False,
)
print("payment rate by BonusMalusBand:")
print(by_bonusmalus.round(3))
print()
print("payment rate by DrivAgeBand:")
print(by_drivage.round(3))

overall payment rate (train): 0.7343

payment rate by BonusMalusBand:


BonusMalusBand
50 (best)    0.645
51-59        0.795
60-79        0.830
80-99        0.819
100-129      0.835
130+         1.000
dtype: float64

payment rate by DrivAgeBand:
DrivAgeBand
18-22    0.889
23-29    0.786
30-39    0.775
40-49    0.734
50-59    0.722
60-69    0.662
70+      0.611
dtype: float64


The share of *reported* claims that actually get *paid* ranges from 0.65 to 1.00 across BonusMalus bands, and falls steadily from 0.89 (youngest drivers) to 0.61 (oldest) across DrivAge bands. A single flat correction factor (0.7343) would under-correct young drivers and over-correct older ones - trading one documented, quantified bias for a smaller, differently-shaped, undocumented one. Rejected in favour of a model that learns this variation directly.

## 2. The resolution: a second frequency model, targeting paid claims

Same formula, same machinery as Phase 5's champion - only the target column changes, from `ClaimNb` (reported) to `ClaimNbFromSev` (paid). Phase 5's champion is not replaced: it remains correct for its own purpose (reported claim frequency). This is a second, purpose-built model specifically for pricing.

In [3]:
paid_freq_result = fit_paid_frequency_glm(train)
print(f"converged: {paid_freq_result.converged}")

paid_freq_pred_train = predict_frequency(paid_freq_result, train)
print(f"predicted total paid claims (train): {paid_freq_pred_train.sum():,.2f}")
print(f"observed total paid claims (train):  {train['ClaimNbFromSev'].sum():,}")
print(f"ratio: {paid_freq_pred_train.sum() / train['ClaimNbFromSev'].sum():.4f}")

converged: True


predicted total paid claims (train): 18,523.00
observed total paid claims (train):  18,523
ratio: 1.0000


**Exact match.** This is not a coincidence or approximation - it's the same canonical-link Poisson GLM property used throughout this project (Phase 5), now applied to the correct target for pricing purposes.

## 3. Combining with severity, and checking the full reconciliation

In [4]:
sev = pd.read_parquet("../data/processed/severity_clean.parquet")
severity_table = build_severity_table(sev, pp_table.drop(columns=["ClaimNbFromSev", "ClaimAmountSum"]))
sev_train = severity_table[severity_table["split"] == "train"].copy()

severity_result, smearing_factor = fit_lognormal_model(sev_train)
# Apply to EVERY policy in the pure-premium table, not just those with claims -
# expected cost IF a claim happens, for every policy regardless of whether it did.
severity_pred_train = predict_lognormal_severity(severity_result, smearing_factor, train)

expected_loss_train = predict_expected_loss(paid_freq_pred_train, severity_pred_train)
print(f"predicted total expected loss (train): \u20ac{expected_loss_train.sum():,.2f}")
print(f"observed total paid claim cost (train): \u20ac{train['ClaimAmountSum'].sum():,.2f}")
print(f"ratio: {expected_loss_train.sum() / train['ClaimAmountSum'].sum():.4f}")

predicted total expected loss (train): €43,700,398.81
observed total paid claim cost (train): €44,004,810.78
ratio: 0.9931


**0.9931 - a dramatic improvement over the original 36.3% overstatement.** The small remaining gap is not a new, unexplained discrepancy: it exactly matches the severity model's own known training-reconciliation imperfection from Phase 6 (0.992), which is expected because Lognormal (like Negative Binomial in Phase 5) doesn't share Poisson's exact-match guarantee.

## 4. Two different numbers that must not be confused

The plan explicitly warns against this exact mistake: **expected loss over a policy's own exposure period** and **annual pure premium** are numerically identical only when `Exposure = 1.0` - for any shorter policy, they diverge, and using the wrong one systematically under-prices partial-year policies.

In [5]:
annual_pp_train = predict_annual_pure_premium(paid_freq_pred_train, severity_pred_train, train["Exposure"])

example = pd.DataFrame({
    "Exposure": train["Exposure"],
    "expected_loss": expected_loss_train,
    "annual_pure_premium": annual_pp_train,
}).sort_values("Exposure").head(3)
print("short-exposure policies: expected_loss is much smaller than annual_pure_premium")
print(example)

short-exposure policies: expected_loss is much smaller than annual_pure_premium
       Exposure  expected_loss  annual_pure_premium
57535  0.002732       0.316824           115.957406
57453  0.002732       0.427402           156.429057
57460  0.002732       0.544031           199.115200


In [6]:
exposure_weighted_annual_pp = (annual_pp_train * train["Exposure"]).sum() / train["Exposure"].sum()
print(f"exposure-weighted average annual pure premium (train): \u20ac{exposure_weighted_annual_pp:.2f}")
print(f"Phase 3's raw empirical estimate (no model): \u20ac167.18")
print(f"difference: {(exposure_weighted_annual_pp/167.18 - 1)*100:.1f}%")

exposure-weighted average annual pure premium (train): €174.19
Phase 3's raw empirical estimate (no model): €167.18
difference: 4.2%


€174.19/policy-year vs. Phase 3's raw empirical €167.18 - close (within ~4%), which is the right sanity check here: a model-based estimate built from two fitted GLMs should land near the raw observed figure, not match it exactly (each model has its own small, already-documented imperfection).

## 5. Summary of step 1

- The 36.3% frequency/severity mismatch documented since Phase 3 is now resolved, not   deferred further.
- A flat correction factor was considered and rejected using real evidence: the "payment   rate" varies substantially by BonusMalus (0.65-1.00) and DrivAge (0.61-0.89).
- Instead, a second frequency model (paid-claim frequency, `ClaimNbFromSev`) was fit using   the exact same formula and machinery as Phase 5's champion - Phase 5's original model is   not replaced, it serves a different purpose.
- Verified: frequency alone reconciles exactly (1.0000); the full combined expected loss   reconciles to 0.9931, with the small remaining gap fully explained by severity's own   already-documented imperfection.
- `predict_expected_loss` and `predict_annual_pure_premium` are kept as two distinct,   clearly-labeled functions to avoid the specific confusion the plan warns against.
- The resulting annual pure premium estimate (\u20ac174.19/policy-year) lands close to Phase   3's raw empirical figure (\u20ac167.18), the expected sanity-check relationship.

Full write-up: `reports/pure_premium_reconciliation.md`.

### Next
Step 2: a direct Tweedie GLM - a single model predicting pure premium directly, without splitting frequency and severity at all - as an alternative to compare against.